<a href="https://colab.research.google.com/github/vijaysingh60616-sudo/vijaysingh60616-sudo.github.io/blob/main/Boosting_math2%261.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 7 Boosting MAth 1: AdaBoost
 Derive and write **sample re-weighting loop; benchmark on imbalanced dataset**

# New SectionAdaBoost (Adaptive Boosting) is an ensemble machine learning algorithm that builds a highly accurate model by combining multiple "weak" learners (typically shallow decision trees, or "stumps").

It works sequentially. It starts by training a weak model on the dataset where all samples have equal importance (weights). After the first model makes its predictions, AdaBoost evaluates the errors. It then increases the weights of the misclassified samples and decreases the weights of the correctly classified ones.

The next weak model is trained on this newly weighted dataset, forcing it to pay closer attention to the difficult examples the previous model got wrong. This re-weighting loop continues for a set number of iterations. The final prediction is a weighted majority vote of all the weak models, where models with lower error rates are given a higher say in the final vote.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Generate an imbalanced dataset (90% class 0, 10% class 1)
X, y = make_classification(n_samples=1000, n_features=20, weights=[0.9, 0.1], random_state=42)

# Convert labels from 0/1 to -1/1 for AdaBoost mathematical formulation
y = np.where(y == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. AdaBoost Sample Re-weighting Loop
n_estimators = 50
n_samples = X_train.shape[0]

# Initialize weights equally: w_i = 1/N
weights = np.ones(n_samples) / n_samples

models = []
alphas = []

for m in range(n_estimators):
    # Fit a weak learner (Decision Stump) using current sample weights
    stump = DecisionTreeClassifier(max_depth=1, random_state=42)
    stump.fit(X_train, y_train, sample_weight=weights)
    predictions = stump.predict(X_train)

    # Calculate weighted error for this stump
    misclassified = (predictions != y_train)
    error = np.sum(weights[misclassified]) / np.sum(weights)

    # Add a small epsilon to prevent division by zero in perfect classification
    error = max(error, 1e-10)

    # Calculate the amount of say (alpha) for this stump
    # alpha = 0.5 * ln((1 - error) / error)
    alpha = 0.5 * np.log((1 - error) / error)

    # Update sample weights:
    # w_i = w_i * exp(-alpha * y_true * y_pred)
    weights = weights * np.exp(-alpha * y_train * predictions)

    # Normalize weights so they sum to 1
    weights /= np.sum(weights)

    # Save the model and its weight
    models.append(stump)
    alphas.append(alpha)

# 3. Benchmark / Evaluation Function
def adaboost_predict(X, models, alphas):
    # Get predictions from all stumps shape: (n_estimators, n_samples)
    stump_preds = np.array([model.predict(X) for model in models])

    # Take the weighted majority vote
    weighted_preds = np.dot(alphas, stump_preds)
    return np.sign(weighted_preds)

y_pred = adaboost_predict(X_test, models, alphas)

print("Custom AdaBoost Benchmark on Imbalanced Data:")
print("--")
print(classification_report(y_test, y_pred))

# Gradient Boosting Mathematical FormulationHere is the step-by-step mathematical breakdown for a standard Gradient Boosting algorithm.1. Initialize the ModelWe start with a simple constant value (like the mean of the target variable) that minimizes the initial loss function $L$:$$F_0(x) = \arg\min_{\gamma} \sum_{i=1}^n L(y_i, \gamma)$$2. The Iterative Loop (for $m = 1$ to $M$ trees)For each iteration $m$, we perform the following sequence:Compute the Pseudo-Residuals: Calculate the negative gradient of the loss function with respect to the previous model's predictions. These act as the "errors" the next tree needs to fix.$$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F(x)=F_{m-1}(x)}$$(Note: If using Mean Squared Error as the loss function, this derivative simplifies exactly to the standard residual: $y_i - F_{m-1}(x_i)$)Fit a Weak Learner: Train a new decision tree (or "stump") $h_m(x)$ to predict these residuals $(x_i, r_{im})$ instead of the original target $y_i$.Calculate the Multiplier: Find the optimal step size $\gamma_m$ for the new tree's predictions that minimizes the overall loss:$$\gamma_m = \arg\min_{\gamma} \sum_{i=1}^n L(y_i, F_{m-1}(x_i) + \gamma h_m(x_i))$$Update the Model: Add the new tree to the existing sequence of models. We apply a learning rate $\nu$ (a small value between 0 and 1) to shrink the tree's contribution and prevent overfitting:$$F_m(x) = F_{m-1}(x) + \nu \cdot \gamma_m h_m(x)$$3. Final OutputAfter $M$ iterations, the final model is the sum of the initial constant and all the incrementally scaled trees:$$F_M(x) = F_0(x) + \sum_{m=1}^M \nu \cdot \gamma_m h_m(x)$$

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X, y = make_classification(n_samples=1000, n_features=20, weights=[0.9, 0.1], random_state=42)

# keep standard 0 and 1 labels for this implementation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

n_estimators = 100
learning_rate = 0.1

f0 = np.mean(y_train)
F_train = np.full(len(y_train), f0)
F_test = np.full(len(y_test), f0)
trees = []

for m in range(n_estimators):
    residuals = y_train - F_train

    # Use DecisionTreeRegressor instead of DecisionTreeClassifier to fit continuous residuals
    tree = DecisionTreeRegressor(max_depth=2, random_state=42)
    tree.fit(X_train, residuals)
    F_train += learning_rate * tree.predict(X_train)
    F_test += learning_rate * tree.predict(X_test)
    trees.append(tree)

y_pred_gb = (F_test >= 0.5).astype(int)

print("Gradient Boosting Benchmark on Imbalanced Data:")
print("--")
print(classification_report(y_test, y_pred_gb))

**Transitioning from ground-up mathematics to production-ready libraries is a crucial pivot for advanced machine learning roles, especially as you prepare for technical interviews this December. The industry standard for this algorithm is XGBoost (eXtreme Gradient Boosting).

1. XGBoost takes the foundational math you just implemented and optimizes it heavily for speed and performance. It introduces three major upgrades:

2. Regularization: It adds L1 and L2 penalties to the loss function, controlling tree complexity to prevent overfitting.

3. Second-Order Gradients: Instead of just using the first derivative (the residual), it uses the second derivative (the Hessian) to take more accurate optimization steps.

Native Imbalance Handling: It allows you to directly pass a weight ratio to force the model to care more about the minority class.**

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report

scale_weight  = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=2,
    scale_pos_weight=scale_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Benchmark with class Weighting:")
print("--")
print(classification_report(y_test, y_pred_xgb))

# The XGBoost Objective FunctionAt step $t$, the goal is to add a new tree $f_t$ that minimizes the following objective function:$$Obj^{(t)} = \sum_{i=1}^n l(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t)$$$l(y_i, \hat{y}_i)$: A differentiable convex loss function (like LogLoss) measuring the difference between the target and the prediction.$\Omega(f_t)$: The regularization term that penalizes the complexity of the new tree to prevent overfitting.The Regularization TermXGBoost defines the complexity of a tree based on its number of leaves and the magnitude of its leaf weights:$$\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2$$$T$: The total number of leaves in the tree.$w_j$: The output score (weight) of the $j$-th leaf.$\gamma$: The penalty parameter for adding a new leaf (controls pruning).$\lambda$: The L2 regularization penalty on leaf weights.Second-Order Taylor ExpansionTo optimize this quickly, XGBoost approximates the objective using a second-order Taylor expansion. It computes both the gradient (first derivative) and the Hessian (second derivative) of the loss function for every data point:Gradient ($g_i$): $g_i = \partial_{\hat{y}^{(t-1)}} l(y_i, \hat{y}^{(t-1)})$Hessian ($h_i$): $h_i = \partial^2_{\hat{y}^{(t-1)}} l(y_i, \hat{y}^{(t-1)})$This simplifies the objective function to focus only on the new tree:$$Obj^{(t)} \approx \sum_{i=1}^n [g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i)] + \Omega(f_t)$$Calculating Optimal Leaf WeightsBy grouping the data points into their respective leaves (where $I_j$ is the set of indices of data points in leaf $j$), we can solve for the optimal weight $w_j^\ast$ of each leaf by taking the derivative with respect to $w_j$ and setting it to zero:$$w_j^\ast = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$Notice how the L2 penalty $\lambda$ sits in the denominator. If $\lambda$ is large, it shrinks the leaf weights towards zero, heavily regularizing the model.The Structure ScoreFinally, substituting $w_j^\ast$ back into the objective gives the scoring function used to evaluate how good a particular tree structure is (lower is better):$$Obj^\ast = -\frac{1}{2} \sum_{j=1}^T \frac{(\sum_{i \in I_j} g_i)^2}{\sum_{i \in I_j} h_i + \lambda} + \gamma T$$When XGBoost builds a tree, it tests different feature splits and uses this exact formula to calculate the "Gain" of a split. If the Gain is less than $\gamma$, it refuses to make the split—this is how it prunes the tree natively.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# 1. Define the grid of hyperparameters to search
param_grid = {
    'max_depth': [2, 3, 5],
    'learning_rate': [0.01, 0.1, 0.2],
    'gamma': [0, 0.1, 0.5] # The exact penalty term from the objective function!
}

# Calculate class weight again
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# 2. Initialize the base model (without the deprecated label encoder argument)
xgb_base = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='logloss'
)

# 3. Initialize GridSearchCV
# cv=3 means 3-fold cross-validation. scoring='f1' tells it what metric to maximize.
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=1
)

# 4. Run the search (this trains 3 * 3 * 3 = 27 models, times 3 folds = 81 fits!)
grid_search.fit(X_train, y_train)

# 5. Extract and evaluate the best model
print("Best Parameters found:", grid_search.best_params_)

best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print("\nTuned XGBoost Benchmark:")
print("--")
print(classification_report(y_test, y_pred_tuned))

# New Section1. LightGBM (Lightning-Fast Speed)

Histogram-Based Splitting: Instead of evaluating every single data point to find the perfect split (which XGBoost traditionally did), LightGBM buckets continuous feature values into discrete bins (histograms). This drastically reduces memory usage and speeds up training.

Leaf-Wise Growth: Standard trees grow level-by-level (depth-wise). LightGBM grows leaf-wise, choosing to split the single leaf that yields the maximum loss reduction, regardless of its level. This converges much faster but makes it prone to overfitting on small datasets.

2. CatBoost (Categorical Mastery)

 . Native Categorical Handling: You do not need to One-Hot Encode or Label Encode categorical data. CatBoost uses a technique called Ordered Target Encoding, which replaces categorical values with a blend of the target variable's expected value, calculated dynamically to prevent data leakage.

 . Symmetric (Oblivious) Trees: CatBoost builds trees where all nodes at the same depth use the exact same feature and split condition. This acts as a heavy regularizer to prevent overfitting and makes predicting on new data incredibly fast.

. Plotting the XGBoost Graphs
To wrap up your XGBoost implementation, visualizing what the model actually learned is the most important step for interpretability. We will plot two graphs: the Feature Importance (to see which features drove the predictions) and an actual Decision Tree from the ensemble.

In [ ]:
import matplotlib.pyplot as plt
import xgboost as xgb

# 1. Plot Feature Importance
# This shows which of your 20 synthetic features were most valuable to the model
plt.rcParams['figure.figsize'] = (10, 6)
xgb.plot_importance(best_model, max_num_features=10, importance_type='weight', title='Top 10 Feature Importances')
plt.show()

# 2. Plot a Single Decision Tree
# This visualizes the very first tree (index 0) built in your tuned ensemble
plt.rcParams['figure.figsize'] = (20, 10)
xgb.plot_tree(best_model, num_trees=0)
plt.title('Tree 0 Structure')
plt.show()

In [ ]:
import lightgbm as lgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Generate an imbalanced dataset
X, y = make_classification(n_samples=1000, n_features=20, weights=[0.9, 0.1], random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Calculate numeric scale_pos_weight ratio
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    random_state=42
)

lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)

print("LightGBM Benchmark with class Weighting:")
print("--")
print(classification_report(y_test, y_pred_lgb))

In [ ]:
!pip install catboost

from catboost import CatBoostClassifier
from sklearn.metrics import classification_report

# Calculate numeric scale_pos_weight ratio
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

cat_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    random_state=42,
    verbose=0
)

cat_model.fit(X_train, y_train)
y_pred_cat = cat_model.predict(X_test)

print("CatBoost Benchmark with class Weighting:")
print("--")
print(classification_report(y_test, y_pred_cat))